# 문항 1 : 네이버 주식 Raw 적재와 Clean 정제

## 크롤링 4번째 과제2에서 작성한 코드를 활용해 네이버 주식 데이터를 Raw 테이블에 저장하고, 데이터를 추출하여 정제 후 Clean 테이블에 적재하시오.

네이버 주식용 테이블 tb_nf_stock 과 금융위 주식용 테이블 tb_fsc_stock 로 각각 테이블을 생성하고 적재하시오.

스키마는 둘 다 공통적으로bas_dt(DATE) / srtn_cd(CHAR 6) / itms_nm / clpr / vs / mkp / hipr / lopr / trqu / raw_id 를 가진다.

일자 / 종목코드 / 종목명 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량

Naver Finance 쪽의 종목명은 NULL이다. 현재 테이블 구조를 어떻게 수정하면 좋을지 주석으로 작성하시오.

금융위 데이터는 Day 8에서 이미 Raw에 적재되어 있다. 네이버만 추가로 수집·적재할 것 (source: naver_finance)

### 정제 요구사항

날짜: 2024.12.24 와 20241224 를 모두 DATE 타입으로

숫자: 쉼표 제거 후 정수형. '11,559,385' → 11559385

전일비: '상승 900' → 900, '하락 500' → -500, '보합 0' → 0

종목코드: 6자리 문자열로 통일

raw_id 를 함께 들고 갈 것

to_sql 을 쓰지 말 것. 테이블을 먼저 만들어 두고, Raw 를 배치 단위로 읽어 정제한 뒤 executemany 로 넣을 것

정제 후 정제 결과를 요약 출력할 것. 변환 건수 / 기간 / 종목수 / 중복률

#### 결과 예시

2026-08-14 10:56:08,401 [INFO] [CLEAN:nf] Raw 400건 → tb_nf_stock (배치 500)
2026-08-14 10:56:08,427 [INFO]   적재 400 / 400
2026-08-14 10:56:08,429 [INFO] ─────── 정제 요약 : nf → tb_nf_stock ───────
2026-08-14 10:56:08,430 [INFO] 변환 건수      읽음 400 · 적재 400
2026-08-14 10:56:08,431 [INFO] 기간           2026-05-19 ~ 2026-07-14
2026-08-14 10:56:08,433 [INFO] 종목수         10
2026-08-14 10:56:08,433 [INFO] 중복률         0.00% (중복 제거 0건)
2026-08-14 10:56:08,457 [INFO] [CLEAN:fsc] Raw 2420건 → tb_fsc_stock (배치 500)
2026-08-14 10:56:08,486 [INFO]   적재 500 / 2420
2026-08-14 10:56:08,520 [INFO]   적재 1000 / 2420
2026-08-14 10:56:08,549 [INFO]   적재 1500 / 2420
2026-08-14 10:56:08,579 [INFO]   적재 2000 / 2420
2026-08-14 10:56:08,605 [INFO]   적재 2420 / 2420
2026-08-14 10:56:08,607 [INFO] ─────── 정제 요약 : fsc → tb_fsc_stock ───────
2026-08-14 10:56:08,607 [INFO] 변환 건수      읽음 2420 · 적재 2420
2026-08-14 10:56:08,609 [INFO] 기간           2025-01-02 ~ 2025-12-30
2026-08-14 10:56:08,610 [INFO] 종목수         10
2026-08-14 10:56:08,611 [INFO] 중복률         0.00% (중복 제거 0건)

In [ ]:
CREATE DATABASE IF NOT EXISTS fsc_db;
USE fsc_db;

CREATE TABLE IF NOT EXISTS tb_nf_stock (
    raw_id INT AUTO_INCREMENT PRIMARY KEY COMMENT '고유ID',
    bas_dt DATE NOT NULL COMMENT '기준일자',
    srtn_cd CHAR(6) NOT NULL COMMENT '종목코드',
    itms_nm VARCHAR(100) COMMENT '종목명',
    clpr INT COMMENT '종가',
    vs VARCHAR(30) COMMENT '전일비',
    mkp INT COMMENT '시가',
    hipr INT COMMENT '고가',
    lopr INT COMMENT '저가',
    trqu BIGINT COMMENT '거래량',
    INDEX idx_srtn_bas (srtn_cd, bas_dt)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

In [1]:
import time
import logging
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import os
import pymysql
from dotenv import load_dotenv

In [11]:
# =====================================================================
# [기본 설정] 로깅 포맷 세팅
# =====================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s,%(msecs)03d [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("ETL")

# 수집 대상 종목코드 및 종목명 매핑 (금융위 수집 데이터와 동일한 코드)
TARGET_STOCKS = {
    "005930": "삼성전자", "000660": "SK하이닉스", "035420": "NAVER", 
    "051910": "LG화학", "005380": "현대차", "006400": "삼성SDI", 
    "035720": "카카오", "068270": "셀트리온", "105560": "KB금융", "055550": "신한지주"
}

In [12]:
# =====================================================================
# [정제 함수 선언] 데이터 정제 요구사항
# =====================================================================

def clean_date(date_text):
    """[요구사항 1] '2024.12.24' 또는 '20241224' 포맷을 모두 YYYY-MM-DD 스트링/DATE 타입으로 변경"""
    if not date_text:
        return None
    clean_txt = date_text.strip().replace(".", "")
    try:
        return datetime.strptime(clean_txt, "%Y%m%d").date()
    except ValueError:
        return None

def clean_int(text):
    """[요구사항 2] 쉼표 제거 후 정수형 변환"""
    if not text:
        return 0
    clean_txt = str(text).replace(",", "").strip()
    return int(clean_txt) if clean_txt.lstrip('-').isdigit() else 0

def clean_vs(text):
    """[요구사항 3] '상승 900' -> 900, '하락 500' -> -500, '보합 0' -> 0 정수 정제"""
    if not text:
        return 0
    text_str = str(text).strip()

    parts = text_str.split()
    if not parts:
        return 0
    if len(parts) == 1:
        return clean_int(parts[0])
        
    keyword, val_str = parts[0], parts[1]
    value = clean_int(val_str)
    
    if "하락" in keyword:
        return -value
    elif "상승" in keyword:
        return value
    else:
        return 0

def clean_srtn_cd(code):
    """[요구사항 4] 종목코드를 6자리 문자열로 통일 (앞자리 0 채움 공정 포함)"""
    if not code:
        return "000000"
    return str(code).strip().zfill(6)

In [19]:
# =====================================================================
# [데이터 크롤링 함수] (기존 소스 원형 유지)
# =====================================================================
def fetch_raw_data():
    url = "https://finance.naver.com/item/sise_day.naver?"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
        }

    # 목표 수집 범위 정의
    START_DATE = clean_date("20250101")
    END_DATE = clean_date("20251231")
    
    raw_data_list = []
    p_raw_id = 1
    
    for srtn_cd in TARGET_STOCKS.keys():
        page = 1
        while True:
            try:
                res = requests.get(url, headers=headers, params={"code": srtn_cd, "page": page})
                if res.status_code != 200: break
            except Exception:
                break
                
            soup = BeautifulSoup(res.text, "html.parser")
            rows = soup.select('table.type2 tr[onMouseOver="mouseOver(this)"]')
            if not rows: break
            
            out_of_range = False
            page_added_cnt = 0
            
            for row in rows:
                tds = row.select("td")
                if len(tds) < 7 or not tds[0].get_text(strip=True): 
                    continue
                
                raw_date_str = tds[0].get_text(strip=True)
                current_date = clean_date(raw_date_str)
                
                if current_date is None:
                    continue
                if current_date > END_DATE:
                    continue
                if current_date < START_DATE:
                    out_of_range = True
                    break
                
                raw_data_list.append({
                    "고유ID": p_raw_id,
                    "날짜": raw_date_str,
                    "종목코드": srtn_cd,
                    "종가": tds[1].get_text(strip=True),
                    "전일비": tds[2].get_text(strip=True),
                    "시가": tds[3].get_text(strip=True),
                    "고가": tds[4].get_text(strip=True),
                    "저가": tds[5].get_text(strip=True),
                    "거래량": tds[6].get_text(strip=True)
                })
                p_raw_id += 1
                page_added_cnt += 1
            
            if out_of_range:
                break
                
            if page_added_cnt == 0 and current_date < START_DATE:
                break
                
            page += 1
            time.sleep(0.15)
            
    return raw_data_list

In [ ]:
# =====================================================================
# [ETL CORE] 정제 및 배치 적재 프로세스
# =====================================================================
def process_etl_batch(raw_data_list, batch_size=200):
    """[요구사항 6] to_sql 미사용, 배치 단위로 유효성 정제 후 executemany 일괄 실행"""

    if not raw_data_list:
        logger.warning("적재할 원천 데이터가 존재하지 않습니다.")
        return

    total_raw_count = len(raw_data_list)
    logger.info(f"[CLEAN:nf] Raw {total_raw_count}건 → tb_nf_stock (배치 {batch_size})")

    
    conn = pymysql.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        charset='utf8mb4',
        autocommit=False
    )
    cursor = conn.cursor()


    insert_query = """
        INSERT INTO tb_nf_stock (raw_id, bas_dt, srtn_cd, itms_nm, clpr, vs, mkp, hipr, lopr, trqu)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    cleaned_batch = []
    inserted_cnt = 0
    
    unique_keys = set()
    duplicate_records_cnt = 0
    min_date, max_date = None, None
    distinct_stocks = set()

    for idx, raw in enumerate(raw_data_list, start=1):
        
        b_dt = clean_date(raw["날짜"])
        s_cd = clean_srtn_cd(raw["종목코드"])
        
        dup_check_key = (b_dt, s_cd)
        if dup_check_key in unique_keys:
            duplicate_records_cnt += 1
            continue
        else:
            unique_keys.add(dup_check_key)

        
        if b_dt:
            if min_date is None or b_dt < min_date: min_date = b_dt
            if max_date is None or b_dt > max_date: max_date = b_dt
        distinct_stocks.add(s_cd)

        
        cleaned_tuple = (
            raw["고유ID"],                              # raw_id 전달
            b_dt,                                       # bas_dt
            s_cd,                                       # srtn_cd
            TARGET_STOCKS.get(s_cd, "알 수 없음"),        # itms_nm (매핑 테이블 결합)
            clean_int(raw["종가"]),                     # clpr
            clean_vs(raw["전일비"]),                     # vs 숫자화 정제 완료
            clean_int(raw["시가"]),                     # mkp
            clean_int(raw["고가"]),                     # hipr
            clean_int(raw["저가"]),                     # lopr
            clean_int(raw["거래량"])                    # trqu
        )
        cleaned_batch.append(cleaned_tuple)

        
        if len(cleaned_batch) == batch_size or idx == total_raw_count:
            if cleaned_batch:
                try:
                    cursor.executemany(insert_query, cleaned_batch)
                    inserted_cnt += len(cleaned_batch)
                    logger.info(f"  적재 {inserted_cnt} / {total_raw_count}")
                    cleaned_batch = []
                except Exception as e:
                    conn.rollback()
                    logger.error(f"배치 적재 도중 예외 발생으로 데이터가 롤백되었습니다: {e}")
                    raise e

    conn.commit()
    cursor.close()
    conn.close()

    # =====================================================================
    # [요구사항 7] 정제 결과 요약 리포트 메인 스크립트 출력 파트
    # =====================================================================
    dup_ratio = (duplicate_records_cnt / total_raw_count) * 100 if total_raw_count > 0 else 0.0
    date_range_str = f"{min_date} ~ {max_date}" if min_date and max_date else "N/A"

    logger.info("─────── 정제 요약 : nf → tb_nf_stock ───────")
    logger.info(f"변환 건수      읽음 {total_raw_count} · 적재 {inserted_cnt}")
    logger.info(f"기간           {date_range_str}")
    logger.info(f"종목수         {len(distinct_stocks)}")
    logger.info(f"중복률         {dup_ratio:.2f}% (중복 제거 {duplicate_records_cnt}건)")

In [21]:
if __name__ == "__main__":
    logger.info("네이버 금융 시세 원천 데이터 수집을 시작합니다...")
    raw_data = fetch_raw_data()
    
    process_etl_batch(raw_data, batch_size=200)

2026-08-28 15:52:35,625 [INFO] 네이버 금융 시세 원천 데이터 수집을 시작합니다...
2026-08-28 15:53:50,713 [INFO] [CLEAN:nf] Raw 2420건 → tb_nf_stock (배치 200)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_40752\1982054491.py:5: DeprecationWarning: 'db' is deprecated, use 'database'
  process_etl_batch(raw_data, batch_size=200)
2026-08-28 15:53:50,744 [INFO]   적재 200 / 2420
2026-08-28 15:53:50,750 [INFO]   적재 400 / 2420
2026-08-28 15:53:50,755 [INFO]   적재 600 / 2420
2026-08-28 15:53:50,760 [INFO]   적재 800 / 2420
2026-08-28 15:53:50,769 [INFO]   적재 1000 / 2420
2026-08-28 15:53:50,775 [INFO]   적재 1200 / 2420
2026-08-28 15:53:50,781 [INFO]   적재 1400 / 2420
2026-08-28 15:53:50,789 [INFO]   적재 1600 / 2420
2026-08-28 15:53:50,796 [INFO]   적재 1800 / 2420
2026-08-28 15:53:50,803 [INFO]   적재 2000 / 2420
2026-08-28 15:53:50,811 [INFO]   적재 2200 / 2420
2026-08-28 15:53:50,818 [INFO]   적재 2400 / 2420
2026-08-28 15:53:50,819 [INFO]   적재 2420 / 2420
2026-08-28 15:53:50,821 [INFO] ─────── 정제 요약 : nf → tb_nf_stock 

# 문항 2 : 분석 지표 설계와 Mart 적재

## tb_fsc_stock 테이블에서 질문에 바로 답하는 형태의 Mart 테이블 두 개를 만드시오.

(1) tb_mart_stock_monthly — 종목 × 월 집계
기준: 종목코드 · 연월(YYYY-MM)

집계 항목

평균 종가 / 최고 종가 / 최저 종가 / 월초 종가 / 월말 종가

거래량 합계 / 거래량 평균

거래일 수 (평균의 신뢰도를 판단하기 위해)

거래일 수가 10일 미만인 달은 제외할 것 (신규 상장·상장폐지 달)

(2) tb_mart_stock_daily — 파생 지표 추가 테이블
추가 파생 지표

chg_pct — 전일 대비 변동률(%)

ma5 · ma20 — 5일·20일 이동평균

vol_ratio — 당일 거래량 ÷ 20일 평균 거래량

반드시 종목별로 그룹화 한 뒤 계산할 것

계산 전에 (종목코드, 날짜) 순으로 정렬할 것

In [ ]:
CREATE TABLE IF NOT EXISTS tb_mart_stock_monthly (
    srtnCd CHAR(6) NOT NULL COMMENT '종목코드',
    yr_mon CHAR(7) NOT NULL COMMENT '날짜(연월)',
    avg_clpr DECIMAL(18,2) COMMENT '평균종가',
    max_clpr INT COMMENT '최고종가',
    min_clpr INT COMMENT '최저종가',
    first_clpr INT COMMENT '월초종가',
    last_clpr INT COMMENT '월말종가',
    sum_trqu BIGINT COMMENT '거래량합계',
    avg_trqu DECIMAL(18,2) COMMENT '거래량평균',
    trade_days INT COMMENT '거래일수',
    PRIMARY KEY (srtnCd, yr_mon)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

In [ ]:
import pymysql

def create_monthly_mart():
    conn = pymysql.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME"),
            charset='utf8mb4',
            autocommit=False
        )
    cursor = conn.cursor()

    JSON_COL = "payload" 

    date_fmt = "%Y-%m"

    sql = f"""
        INSERT INTO tb_mart_stock_monthly (
            srtnCd, yr_mon, avg_clpr, max_clpr, min_clpr, 
            first_clpr, last_clpr, sum_trqu, avg_trqu, trade_days
        )
        SELECT 
            JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.srtnCd')) AS srtnCd,
            
            -- [수정] %% 문법 대신 파이썬 변수 매핑 기법을 적용하여 안전하게 포맷팅 전달
            DATE_FORMAT(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt')), '{date_fmt}') AS yr_mon,
            
            AVG(CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.clpr')) AS UNSIGNED)) AS avg_clpr,
            MAX(CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.clpr')) AS UNSIGNED)) AS max_clpr,
            MIN(CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.clpr')) AS UNSIGNED)) AS min_clpr,
            
            MAX(CASE WHEN JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt')) = m.first_date 
                     THEN CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.clpr')) AS UNSIGNED) END) AS first_clpr,
            MAX(CASE WHEN JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt')) = m.last_date 
                     THEN CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.clpr')) AS UNSIGNED) END) AS last_clpr,
            
            SUM(CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.trqu')) AS UNSIGNED)) AS sum_trqu,
            AVG(CAST(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.trqu')) AS UNSIGNED)) AS avg_trqu,
            
            COUNT(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt'))) AS trade_days
        FROM tb_fsc_stock t
        INNER JOIN (
            SELECT 
                JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) AS srtnCd,
                DATE_FORMAT(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')), '{date_fmt}') AS yr_mon,
                MIN(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt'))) AS first_date,
                MAX(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt'))) AS last_date
            FROM tb_fsc_stock
            GROUP BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')), DATE_FORMAT(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')), '{date_fmt}')
        ) m ON JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.srtnCd')) = m.srtnCd 
           AND DATE_FORMAT(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt')), '{date_fmt}') = m.yr_mon
        GROUP BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')), DATE_FORMAT(JSON_UNQUOTE(JSON_EXTRACT(t.{JSON_COL}, '$.basDt')), '{date_fmt}')
        HAVING trade_days >= 10;
    """
    
    try:
        cursor.execute("TRUNCATE TABLE tb_mart_stock_monthly;")
    
        cursor.execute(sql)
        conn.commit()
        
        print(f"생성 및 적재된 마트 요약 데이터 총 건수: {cursor.rowcount}건")
        
    except Exception as e:
        conn.rollback()
        print(f"[오류] 월별 마트 데이터 적재 실패: {e}")
        
    finally:
        cursor.close()
        conn.close()

# 함수 가동
create_monthly_mart()

생성 및 적재된 마트 요약 데이터 총 건수: 120건


In [ ]:
CREATE TABLE IF NOT EXISTS tb_mart_stock_daily (
    basDt DATE NOT NULL COMMENT '기준일자',
    srtnCd CHAR(6) NOT NULL COMMENT '종목코드',
    itmsNm VARCHAR(100) COMMENT '종목명',
    clpr INT COMMENT '종가',
    chg_pct DECIMAL(7,2) COMMENT '전일대비변동률(%)',
    ma5 DECIMAL(18,2) COMMENT '5일이동평균',
    ma20 DECIMAL(18,2) COMMENT '20일이동평균',
    vol_ratio DECIMAL(10,4) COMMENT '거래량비율(당일/20일평균)',
    PRIMARY KEY (basDt, srtnCd)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

In [ ]:
import pymysql

def create_daily_mart():
    conn = pymysql.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME"),
            charset='utf8mb4',
            autocommit=False
        )
    cursor = conn.cursor()
    
    JSON_COL = "payload"

    date_in_fmt = "%Y%m%d"
    
    sql = f"""
        INSERT INTO tb_mart_stock_daily (
            basDt, srtnCd, itmsNm, clpr, chg_pct, ma5, ma20, vol_ratio
        )
        SELECT 
            STR_TO_DATE(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')), '{date_in_fmt}') AS basDt,
            JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) AS srtnCd,
            JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.itmsNm')) AS itmsNm,
            CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED) AS clpr,
            
            -- [지표 1] 전일 대비 변동률(%): (금일종가 - 전일종가) / 전일종가 * 100
            ROUND(
                (CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED) - 
                 LAG(CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED), 1) 
                 OVER (PARTITION BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) 
                       ORDER BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')) ASC))
                / LAG(CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED), 1) 
                OVER (PARTITION BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) 
                      ORDER BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')) ASC) * 100
            , 2) AS chg_pct,
            
            -- [지표 2] 5일 이동평균 (종목별 그룹화 후 당일 포함 과거 5일 이동평균 가격산출)
            ROUND(
                AVG(CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED)) 
                OVER (PARTITION BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) 
                      ORDER BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')) ASC 
                      ROWS BETWEEN 4 PRECEDING AND CURRENT ROW)
            , 2) AS ma5,
            
            -- [지표 3] 20일 이동평균 (종목별 그룹화 후 당일 포함 과거 20일 이동평균 가격산출)
            ROUND(
                AVG(CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.clpr')) AS SIGNED)) 
                OVER (PARTITION BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) 
                      ORDER BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')) ASC 
                      ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
            , 2) AS ma20,
            
            -- [지표 4]  당일 거래량 ÷ 20일 평균 거래량 비율
            ROUND(
                CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.trqu')) AS SIGNED) / 
                AVG(CAST(JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.trqu')) AS SIGNED)) 
                OVER (PARTITION BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.srtnCd')) 
                      ORDER BY JSON_UNQUOTE(JSON_EXTRACT({JSON_COL}, '$.basDt')) ASC 
                      ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
            , 4) AS vol_ratio
        FROM tb_fsc_stock
        ORDER BY srtnCd ASC, basDt ASC;
    """
    
    try:
        cursor.execute("TRUNCATE TABLE tb_mart_stock_daily;")
        
        cursor.execute(sql)
        conn.commit()
        
        print(f"생성 및 적재된 마트 파생 데이터 총 건수: {cursor.rowcount}건")
        
    except Exception as e:
        conn.rollback()
        print(f"[오류] 일별 파생 마트 연산 및 적재 실패: {e}")
        
    finally:
        cursor.close()
        conn.close()

# 함수 가동
create_daily_mart()


생성 및 적재된 마트 파생 데이터 총 건수: 2420건
